# Density fitting and streaming

The Coulomb backend ladder: exact ERIs for small systems, RI density fitting at O(N^3), and streamed/screened paths when the tensors stop fitting.

In [1]:
import jax; jax.config.update("jax_enable_x64", True)
from dftax import KS, Molecule, becke, df, scf
from dftax.energy.xc import PBE

mol = Molecule.from_xyz("O 0 0 0; H 0.7586 0 0.5043; H 0.7586 0 -0.5043", "sto-3g")
AUX = "def2-universal-jkfit"
e_exact = scf(KS(mol, PBE())).e_tot
e_df = scf(KS(mol, PBE(), coulomb=df(AUX))).e_tot
e_stream = scf(KS(mol, PBE(),
                  coulomb=df(AUX, chunk=64, screen=1e-10),
                  grid=becke(chunk=20_000))).e_tot
print(f"exact         E = {e_exact:.8f}")
print(f"RI-J (DF)     E = {e_df:.8f}   |Δ| = {abs(e_df - e_exact):.1e}")
print(f"streamed+scr  E = {e_stream:.8f}   |Δ| = {abs(e_stream - e_exact):.1e}")

exact         E = -75.14676588
RI-J (DF)     E = -75.14720777   |Δ| = 4.4e-04
streamed+scr  E = -75.14720777   |Δ| = 4.4e-04
